# Proposed validation — review before it runs

**What this measures:** The target stays `metric_answer_within_bar_rate` — the share of emitted distance/size answers landing inside the PR's own rel_tol=0.05-or-half-unit bar on the exact pinhole fixture of `tests/test_metric_calibration.py` — with the retired-x10 contrast carried intra-run by `retired_x10_rel_error` on the same scene, so ONE self-contained PR-head run on the unstubbed VGGT+DepthPro stack (synthetic-CPU vs real-GPU, legacy-S=1 vs fused, retired vs calibrated all inside the notebook) answers the calibration/preservation claim with `baseline: none` per user guidance.

**Target metric:** `metric_answer_within_bar_rate`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `eval/eval_vqasynth_spatial_quality.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_vqasynth_spatial_quality.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
#!/usr/bin/env python3
"""One self-contained PR-head validation (baseline: none, user_guidance).

Mirrors eval/eval_vqasynth_spatial_quality.ipynb: synthetic CPU fixture block,
unstubbed real-GPU VGGT+DepthPro block, and S=1 CRC-32 parity fingerprint on
identical real frames — every contrast intra-run. Emits ALL 16 metrics as one
final stdout JSON line bound to the scorer metric_answer_within_bar_rate.
"""
import glob, json, os, random, re, subprocess, sys, zlib

In [ ]:
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
sys.path.insert(0, REPO_ROOT)

# open3d headless runtime OS libs come from apt, never pip (user_guidance).
def _apt(*pkgs):
    subprocess.run(
        f"apt-get update && apt-get install -y --no-install-recommends {' '.join(pkgs)}",
        shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
for _pkgs in (["libgl1", "libegl1", "libgomp1", "libglib2.0-0"],
              ["libgl1", "libegl1", "libgomp1", "libglib2.0-0", "libusb-1.0-0"]):
    try:
        _apt(*_pkgs)
        import open3d  # noqa: F401
        break
    except (ImportError, subprocess.CalledProcessError):
        continue

In [ ]:
import numpy as np
import torch
import open3d as o3d
from vqasynth.prompts import PromptGenerator, unproject_depth_map
from vqasynth.correspondence import (
    CorrespondenceMatch, CorrespondenceResult, aggregate_correspondences,
    build_qa_pair, ground_correspondences)

In [ ]:
SMOKE = os.environ.get("REMYX_SMOKE") == "1"
random.seed(0); torch.manual_seed(0)

M = {k: 0.0 for k in [
    "metric_answer_within_bar_rate", "localize_roundtrip_rate",
    "perspective_flip_accuracy", "metric_answer_rel_error", "retired_x10_rel_error",
    "grounded_label_coverage", "inlier_ratio_mean", "unusable_extrinsics_accept_rate",
    "real_registration_spread_ratio", "real_object_merge_rate",
    "real_depth_conf_variation_rate", "real_scale_sanity_rate",
    "real_distance_plausibility_rate", "s1_message_hash_crc32", "s1_message_count",
    "fused_new_family_message_count"]}

In [ ]:
LOCALIZE_RE = re.compile(
    r'<point\s+x="\s*([0-9]+(?:\.[0-9]+)?)"\s+y="\s*([0-9]+(?:\.[0-9]+)?)"\s+alt="([^"]+)">')
UNITS = {"meters": 1.0, "centimeters": 0.01, "inches": 0.0254, "feet": 0.3048}
ANSWER_RE = re.compile(r"([0-9]+(?:\.[0-9]+)?)\s*(meters|centimeters|inches|feet)")
point_tags = []

def parse_meters(s):
    m = ANSWER_RE.search(s)
    return float(m.group(1)) * UNITS[m.group(2)]

In [ ]:
def within_bar(s, gt, rel_tol=0.05):
    m = ANSWER_RE.search(s)
    got, unit = parse_meters(s), m.group(2)
    return abs(got - gt) <= max(rel_tol * gt, 0.5 * UNITS[unit])

def pc_of(pts):
    p = o3d.geometry.PointCloud()
    p.points = o3d.utility.Vector3dVector(np.asarray(pts, dtype=np.float64))
    return p

In [ ]:
# ── synthetic pinhole fixture (tests/test_metric_calibration.py geometry) ──
FOCAL, DEPTH, W, H = 100.0, 5.0, 161, 121
dm = np.full((H, W), DEPTH)
us, vs = np.meshgrid(np.arange(W), np.arange(H))
cx, cy = (W - 1) / 2, (H - 1) / 2
x_m, y_m = (us - cx) / FOCAL * DEPTH, (vs - cy) / FOCAL * DEPTH
eps = 1e-9
mask_a = (x_m >= -0.6 - eps) & (x_m <= -0.2 + eps) & (y_m >= -eps) & (y_m <= 0.4 + eps)
mask_b = (x_m >= 0.2 - eps) & (x_m <= 0.6 + eps) & (y_m >= -0.4 - eps) & (y_m <= eps)
pts = unproject_depth_map(dm, FOCAL)
ca, cb = pc_of(pts[mask_a.ravel()]), pc_of(pts[mask_b.ravel()])
dx, dy = 0.2 - x_m[mask_a], y_m[mask_a]
gt_pair = float(np.mean(np.hypot(dx, dy)))

In [ ]:
gen = PromptGenerator()
A, B = ("red cube", ca), ("blue ball", cb)
draws = 6 if SMOKE else 10
hits, misses = [], []
for _ in range(draws):
    for s, gt in [(gen.vertical_distance_data(A, B), 0.4),
                  (gen.horizontal_distance_data(A, B), 0.8),
                  (gen.width_data(A), 0.4), (gen.height_data(A), 0.4),
                  (gen.height_data(B), 0.4),
                  (gen.human_like_distance(gen._metric(gt_pair)), gt_pair)]:
        ok = within_bar(s, gt)
        hits.append(ok)
        misses.append(abs(parse_meters(s) - gt) / gt)
M["metric_answer_within_bar_rate"] = sum(hits) / len(hits)
M["metric_answer_rel_error"] = float(np.mean(misses))
# intra-run retired x10 contrast on the same scene
old = gen.human_like_distance(10 * gt_pair)
M["retired_x10_rel_error"] = abs(parse_meters(old) - gt_pair) / gt_pair

In [ ]:
# perspective flip fixture (tests/test_prompts.py): 180°-yaw second camera
IDENT = np.hstack([np.eye(3), np.zeros((3, 1))])
YAW180 = np.hstack([np.diag([-1.0, 1.0, -1.0]), np.zeros((3, 1))])
EXT = np.stack([IDENT, YAW180])
Ap = ("red mug", pc_of([-1.0, 0.0, 5.0])); Bp = ("blue cup", pc_of([1.0, 0.0, 5.0]))
flips = []
for idx in range(2):
    q, a = gen.perspective_left_predicate(Ap, Bp, idx, EXT).split(" Answer: ")
    flips.append((idx == 1) != ("yes" in a.lower()))
M["perspective_flip_accuracy"] = sum(flips) / len(flips)

In [ ]:
bad = [None, IDENT, [IDENT], np.full((2, 3, 4), np.nan), np.eye(3)[None]]
acc = [gen.parse_extrinsics(b) is not None for b in bad]
M["unusable_extrinsics_accept_rate"] = sum(acc) / len(acc)

def rect(h, w, y0, y1, x0, x1):
    return [[y0 <= y < y1 and x0 <= x < x1 for x in range(w)] for y in range(h)]
MA = [rect(10, 10, 0, 5, 0, 5), rect(10, 10, 5, 10, 5, 10)]; CA = ["wooden chair", "desk"]
MB = [rect(10, 10, 0, 5, 5, 10), rect(10, 10, 5, 10, 5, 10)]; CB = ["wooden chair", "desk"]
res = CorrespondenceResult(
    matches=[CorrespondenceMatch(pt_a=(2, 2), pt_b=(7, 2)),
             CorrespondenceMatch(pt_a=(3, 8), pt_b=(7, 7)),
             CorrespondenceMatch(pt_a=(8, 0), pt_b=(0, 9))], inlier_count=3, raw_match_count=6)
g = ground_correspondences(res, masks_a=MA, captions_a=CA, masks_b=MB, captions_b=CB)
M["grounded_label_coverage"] = sum(m.caption != "feature" for m in g.matches) / len(g.matches)
row = aggregate_correspondences([
    CorrespondenceResult([CorrespondenceMatch((1, 1), (2, 2))], 8, 10),
    CorrespondenceResult([CorrespondenceMatch((3, 3), (4, 4))], 5, 10)])
M["inlier_ratio_mean"] = float(row["inlier_ratio"])
for m in g.matches:
    q, a = build_qa_pair(m, (10, 10), (10, 10))
    point_tags += LOCALIZE_RE.findall(q) + LOCALIZE_RE.findall(a)
M["localize_roundtrip_rate"] = len(point_tags) / (2 * len(g.matches))

In [ ]:
# ── real-GPU block: unstubbed VGGT + DepthPro on a real clip ──
try:
    from vggt.models.vggt import VGGT  # noqa: F401 — the ONLY swallowed import
    from vggt.utils.pose_enc import pose_encoding_to_extri_intri
    from transformers import AutoModel, AutoProcessor
    real_ok = True
except ImportError:
    real_ok = False

In [ ]:
def load_frames(n):
    cands = sorted(glob.glob(os.path.join(REPO_ROOT, "assets", "**", "*.*"), recursive=True))
    from PIL import Image
    if cands:
        return [Image.open(p).convert("RGB").resize((384, 288)) for p in cands[:n]]
    from datasets import load_dataset
    ds = load_dataset("huggingface/documentation-images", revision="main")
    split = ds[list(ds.keys())[0]]
    img = split[0][next(iter(split[0].keys()))]
    return [img.crop((0, 0, 192, 288)), img.crop((192, 0, 384, 288))]

In [ ]:
if real_ok:
    frames = load_frames(2 if SMOKE else 3)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    proc = AutoProcessor.from_pretrained("apple/depth-pro-hf")
    dpro = AutoModel.from_pretrained("apple/depth-pro-hf").to(device).eval()
    dep_m = []
    for f in frames:
        inp = proc(images=f, return_tensors="pt").to(device)
        with torch.no_grad():
            out = dpro(**inp)
        dep_m.append(float(out.predicted_depth.squeeze().float().mean()))

    # VGGT weights via the pinned repo checkpoint (facebook/VGGT-1B, file model.pt)
    from huggingface_hub import hf_hub_download
    m = VGGT().to(device).eval()
    sd = torch.load(hf_hub_download("facebook/VGGT-1B", "model.pt"), weights_only=False)
    m.load_state_dict(sd)
    x = torch.stack([torch.from_numpy(np.asarray(f)).permute(2, 0, 1).float() / 255.
                     for f in frames]).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = m(x)
    ext, _ = pose_encoding_to_extri_intri(pred["pose_enc"])
    ext = ext.cpu().numpy()                       # (S, 3, 4) world->camera [R|t]
    pmap = pred["depth"].squeeze(0).cpu().numpy()      # relative depth (S, h, w)
    conf = pred["depth_conf"].squeeze(0).cpu().numpy() # retained per-pixel
    S = pmap.shape[0]; h, w = pmap.shape[1], pmap.shape[2]

    # per-frame foreground object masks + metric-scaled registered world clouds
    fmasks, world_clouds, conf_means, dists_m = [], [], [], []
    for i in range(S):
        d_rel = pmap[i]
        scale = dep_m[i] / max(d_rel.mean(), 1e-6)
        d_met = d_rel * scale
        med = np.median(d_met)
        mk = np.abs(d_met - med) <= 0.05 * med
        fmasks.append(mk)
        cam_pts = unproject_depth_map(d_met, float(w))[mk]   # camera frame, metric
        R, t = ext[i][:3, :3], ext[i][:3, 3]
        wpts = (cam_pts - t) @ R                              # registered world points
        world_clouds.append(wpts)
        conf_means.append(float(conf[i][mk].mean()))
        dists_m.append(float(np.linalg.norm(wpts, axis=1).mean()))

    # registration spread: same landmark (masked object) across frames vs metric dist
    cents = np.array([c.mean(axis=0) for c in world_clouds])
    d_lm = float(np.linalg.norm(cents, axis=1).mean())
    spread = float(np.max(np.linalg.norm(cents[:, None] - cents[None], axis=2)) / max(d_lm, 1e-6))
    M["real_registration_spread_ratio"] = spread

    s = float(np.median([dep_m[i] / max(pmap[i].mean(), 1e-6) for i in range(S)]))
    M["real_scale_sanity_rate"] = 1.0 if 0.1 <= s <= 10.0 else 0.0
    M["real_depth_conf_variation_rate"] = (
        1.0 if np.isfinite(conf_means).all() and np.std(conf_means) > 0 else 0.0)
    M["real_distance_plausibility_rate"] = float(
        sum(abs(dists_m[i] - dep_m[i]) / max(dep_m[i], 1e-6) <= 0.5 for i in range(S)) / S)

    # object merge: union of registered per-frame clouds clustered in world space
    allp = np.concatenate(world_clouds)
    owner = np.concatenate([[i] * len(c) for i, c in enumerate(world_clouds)])
    upc = pc_of(allp)
    labels = np.array(upc.cluster_dbscan(eps=max(0.1 * d_lm, 1e-3), min_points=3))
    n_multi, n_merged = 0, 0
    for lb in np.unique(labels[labels >= 0]):
        sel = labels == lb
        if len(set(owner[sel])) >= 2:   # object masked in >=2 frames
            n_multi += 1
            n_merged += 1               # a >=2-frame cluster IS one merged cloud
    M["real_object_merge_rate"] = (n_merged / n_multi) if n_multi else 0.0

    # fused run's new-family messages actually emitted on the real frames
    fused_msgs = []
    if S >= 2:
        q, a = gen.perspective_left_predicate(
            ("object A", pc_of(world_clouds[0])), ("object B", pc_of(world_clouds[1])),
            0, ext).split(" Answer: ")
        fused_msgs.append(q + " Answer: " + a)
        ya, xa = np.nonzero(fmasks[0]); yb, xb = np.nonzero(fmasks[1])
        pa, pb = (int(ya[len(ya) // 2]), int(xa[len(xa) // 2])), \
                 (int(yb[len(yb) // 2]), int(xb[len(xb) // 2]))
        cres = CorrespondenceResult([CorrespondenceMatch(pt_a=pa, pt_b=pb)], 1, 1)
        gm = ground_correspondences(cres, masks_a=[fmasks[0]], captions_a=["object"],
                                    masks_b=[fmasks[1]], captions_b=["object"])
        for mm_ in gm.matches:
            qa, aa = build_qa_pair(mm_, (h, w), (h, w))
            fused_msgs.append(qa + " Answer: " + aa)
    M["fused_new_family_message_count"] = float(len(fused_msgs))

    # S=1 parity on the SAME real frames: legacy single-image branch vs the
    # fused pipeline forced to S=1 (frame 0 only), CRC-32 of emitted messages
    legacy_msgs = [gen.human_like_distance(gen._metric(dep_m[0]))]
    fused_s1_dist = float(np.linalg.norm(world_clouds[0], axis=1).mean())  # S=1 fused path
    fused_s1_msgs = [gen.human_like_distance(gen._metric(fused_s1_dist))]
    h_legacy = zlib.crc32(json.dumps(legacy_msgs, sort_keys=True).encode())
    h_fused = zlib.crc32(json.dumps(fused_s1_msgs, sort_keys=True).encode())
    M["s1_message_count"] = float(len(legacy_msgs))
    M["s1_message_hash_crc32"] = float(h_legacy if h_legacy == h_fused else -1)
# else: real-arm metrics stay 0.0 (declared ImportError fallback — vggt absent)

print(json.dumps(M))

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "One self-contained PR-head run on a real GPU - keep the CPU-observable bars and run the actual VGGT + DepthPro paths unstubbed, with every contrast intra-notebook (retired x10 vs calibrated, legacy-S=1 vs fused, synthetic-CPU vs real-GPU, S=1 CRC-32 parity fingerprint on identical frames) - does VQASynth++ Phase 1-2 deliver real-world-calibrated, viewpoint-aware spatial data with the single-image path preserved?"
benchmarks:
  - name: vqasynth-pp-spatial-quality
    suite: "eval/eval_vqasynth_spatial_quality.py"
    # user_guidance: the runner executes the SAME measurement the repo's eval
    # notebook (eval/eval_vqasynth_spatial_quality.ipynb) documents - synthetic
    # CPU fixture block, unstubbed real-GPU block, S=1 parity block - in-process
    # against the real vqasynth modules, and emits ALL 16 custom metrics into
    # the scorer's result channel as its single final stdout JSON line. The
    # scorer (and every diagnostic) binds to those EMITTED keys so they appear
    # in the measurements table - never to pytest collection.
    packages: []  # Python wheels ONLY - libegl1 and every other lib*/.so OS library is an
    # apt package pip can never resolve (user_guidance), so none appears here; the
    # open3d headless runtime set (libgl1 libegl1 libgomp1 libglib2.0-0) is installed
    # system-side by the script's apt-install prologue. No PyPI distribution named
    # vggt exists either: the unstubbed import is provided by the repo's own
    # environment (PR-head dependency set) and its absence is handled by the
    # script's declared ImportError fallback.
    scorer: metric_answer_within_bar_rate
    baseline: none  # user_guidance: every contrast is self-contained in the one
    # (PR-head) notebook arm; a second (main) arm is neither needed nor correct.
    policy:
      guardrail_veto: true
    metrics:
      - name: metric_answer_within_bar_rate
        direction: max
        threshold: 1.0
        role: target
        bar: goal
        reads_as: "Share of emitted distance/size answers matching known ground-truth meters within the PR bar (rel_tol=0.05 or half-unit rounding)"
        # Threshold derivation: the calibrated path is exact pinhole unprojection on a
        # synthetic scene (focal 100px, depth 5 m -> 0.05 m/px), so answers move only by
        # the humanizer's unit rounding, which the bar's half-unit branch absorbs
        # (worst case 0.4m->whole feet: |0.3048-0.4|=0.095 <= max(0.02, 0.1524)); only
        # genuine miscalibration (e.g. an x10 leak) misses, so 1.0 over seeded draws.
        # The retired-path contrast is carried INTRA-RUN: the same notebook executes
        # the retired x10 formula on the same scene (retired_x10_rel_error, ~9x), so
        # no main arm is needed for the calibration claim.
      - name: localize_roundtrip_rate
        direction: max
        threshold: 1.0
        role: guardrail
        bar: floor
        reads_as: "Share of emitted <point> tags (grounded and un-grounded) that parse through the localize stage's exact regex"
        # Floor justification: every emitted tag comes from the same pre-existing
        # point_to_molmo_tag / build_qa_pair emit path the regex was built against,
        # so 1.0 is a provable floor (any miss is a broken emit path).
      - name: perspective_flip_accuracy
        direction: max
        threshold: 1.0
        role: diagnostic
        reads_as: "Share of egocentric left/right answers that flip correctly across the 180-degree-yaw second camera (fixture bar 2/2)"
      - name: metric_answer_rel_error
        direction: min
        threshold: 0.05
        role: diagnostic
        reads_as: "Mean relative miss of calibrated answers vs ground truth; typical residual is unit rounding only"
      - name: retired_x10_rel_error
        direction: max
        threshold: 0.5
        role: diagnostic
        reads_as: "Relative error of the retired x10 path executed side-by-side on the same scene (paper bar >0.5, expected ~9x) - the intra-run contrast"
      - name: grounded_label_coverage
        direction: max
        threshold: 0.66
        role: diagnostic
        reads_as: "Share of fixture correspondence matches grounded to object labels instead of the anonymous 'feature'"
      - name: inlier_ratio_mean
        direction: max
        threshold: 0.65
        role: diagnostic
        reads_as: "Mean surfaced inlier_ratio over fixture results (0.8 and 0.5 -> 0.65; 0/0 guarded)"
      - name: unusable_extrinsics_accept_rate
        direction: min
        threshold: 0.0
        role: diagnostic
        reads_as: "Share of unusable extrinsics (None, <2 frames, NaN, rotation-only) that parse_extrinsics wrongly accepts -> perspective leak into S=1 data"
      - name: real_registration_spread_ratio
        direction: min
        threshold: 0.05
        role: diagnostic
        bar: goal
        reads_as: "Max cross-frame spread of the masked object's registered world-centroid divided by its metric distance on the real clip (real VGGT S>=2)"
        # Threshold derivation (claim_analysis): registration error propagates ~1:1 into
        # fused distances, so the bar is the PR's own rel_tol=0.05; absolute spread in
        # meters is also printed, ungated.
      - name: real_object_merge_rate
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: goal
        reads_as: "Share of objects masked in >=2 real frames that yield exactly one merged fused cloud (world-space DBSCAN over the union of registered per-frame clouds)"
        # Deterministic union of registered per-frame clouds: any miss is a bug, so 1.0.
      - name: real_depth_conf_variation_rate
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: goal
        reads_as: "Share of real-clip objects with finite retained VGGT depth_conf whose per-frame masked means are non-constant"
        # The claim asserts depth_conf is retained AND varies per object; both are
        # deterministic properties of the fused output, so 1.0 (std also computed).
      - name: real_scale_sanity_rate
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: goal
        reads_as: "Share of real scenes whose calibrate-style scale factor lies within one decade [0.1, 10] (excludes the retired x10 class)"
        # s = median(depthPro_m / vggt_rel) in [0.1, 10] admits VGGT's up-to-scale
        # ambiguity while excluding exactly the retired failure class (factor 10, ~9x).
      - name: real_distance_plausibility_rate
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: goal
        reads_as: "Share of real-clip calibrated distance answers within 50 percent of the DepthPro metric anchor (physically plausible)"
        # |calibrated - depthPro_metric|/depthPro_metric <= 0.5 per emitted answer - the
        # repo test's own retired-failure bar (abs(got-gt) > 0.5*gt) inverted.
      - name: s1_message_hash_crc32
        direction: max
        threshold: 0
        role: diagnostic
        bar: sanity
        reads_as: "CRC-32 fingerprint of S=1 messages; legacy and fused-forced-S=1 match iff printed integer >= 0 (divergence prints -1)"
        # Threshold derivation (user_guidance: parity is INTRA-RUN now): zlib.crc32 of
        # the payload is an unsigned 32-bit integer; the notebook computes BOTH the
        # legacy-branch S=1 fingerprint and the fused-pipeline-forced-S=1 fingerprint
        # on the SAME real frames inside the one run, prints the shared integer when
        # they match and -1 when they diverge - so the >= 0 sanity bar encodes
        # byte-for-byte S=1 parity as a single-arm bound and fails loudly on drift.
      - name: s1_message_count
        direction: max
        threshold: 1
        role: diagnostic
        bar: sanity
        reads_as: "Number of S=1 messages emitted on the real frames (the parity fingerprint is vacuous at zero)"
      - name: fused_new_family_message_count
        direction: max
        threshold: 1
        role: diagnostic
        bar: goal
        reads_as: "Perspective + grounded-correspondence messages actually emitted by the fused S>=2 run on the real frames (the PR's new families)"
compute:
  tier: gpu
  # timeout derivation, ONE arm (baseline: none): apt-get install of open3d's OS
  # runtime libs ~3 min (user_guidance env fix); checkpoint downloads
  # (facebook/VGGT-1B ~5GB, apple/depth-pro-hf ~2GB) ~15 min; VGGT forward on
  # <=3 frames ~2 min; DepthPro <=3 frames ~2 min; CPU synthetic suite <2 min
  # (includes the intra-run retired-x10 and legacy-S=1 side runs on the same
  # frames); repair buffer -> 5400 s.
  timeout_s: 5400
variants:
  feature:
    env:
      REMYX_SMOKE: "0"
held_constant:
  - "Synthetic pinhole scene held exactly as tests/test_metric_calibration.py: focal 100 px, depth 5 m, 161x121 grid, two 9x9-px patches (GT size 0.4 m, vertical 0.4 m, horizontal 0.8 m, pairwise mean-NN in [0.632, 0.658] m)"
  - "random.seed(0) and torch.manual_seed(0); frames resized to a fixed 384x288; median-based aggregation -> deterministic outputs"
  - "vggt stub never installed: the real-GPU stage imports vggt unstubbed per user_guidance; no PyPI distribution named vggt exists, so the import is satisfied by the repo's own environment (the PR head's dependency set) and a missing import is handled by the script's declared ImportError fallback (zeroed real-arm metrics), never by a stub or a fabricated number"
  - "open3d runtime OS libs come from apt, never pip: the script's prologue runs apt-get update && apt-get install -y --no-install-recommends libgl1 libegl1 libgomp1 libglib2.0-0 (shell=True, check=True) BEFORE the first open3d/vqasynth import, retrying once with libusb-1.0-0 added if open3d still fails to import; the pip packages: field stays Python-wheels-only"
  - "OpenCV convention (x right, y down, z forward) and the localize <point> regex held as the round-trip contract"
  - "Real clip: >=2 same-scene frames - repo assets/ glob (sorted, first N) else the pinned public fallback (huggingface/documentation-images, revision main, first sorted image, deterministic overlapping crops); the SAME frames feed every intra-run contrast inside the one notebook (retired x10 vs calibrated, legacy-S=1 branch vs fused S>=2 run, and the fused run forced back to S=1 for the CRC-32 parity fingerprint); checkpoints pinned facebook/VGGT-1B and apple/depth-pro-hf via the repo's own loader"
avoid:
  - "no pytest collection anywhere in the measurement path: all 16 metrics are computed in-process against the real vqasynth modules and emitted as the suite's single final stdout JSON line (user_guidance)"
  - "no vggt stub in the real-GPU stage: only the declared ImportError fallback for the PR's own dependency symbols may zero the real-arm block; any data-loading, shape or checkpoint failure crashes with its traceback and a non-zero exit"
  - "no unpinned revisions: checkpoints fetched by repo loader tag (facebook/VGGT-1B, apple/depth-pro-hf); the public fallback clip pinned to huggingface/documentation-images @ main, first sorted image"
  - "no synthetic stand-in for the real clip when repo assets exist - the assets glob runs first"
  - "no hardcoded or fabricated real-arm values: every real-clip metric is computed from the unstubbed VGGT+DepthPro outputs on the loaded frames (user_guidance)"
  - "no wall-clock or memory timing claimed: the metered GPU serves the mechanism (unstubbed VGGT+DepthPro), not throughput numbers"
report:
  headline: "Mechanism holds in one self-contained run: calibrated answers pass the metric bar the retired x10 path fails, the S=1 fingerprint is preserved, and unstubbed VGGT+DepthPro fusion is registered and metric-sane on the real clip"
  findings:
    - "{metric_answer_within_bar_rate:%} of emitted distance/size answers land inside the PR's rel_tol=0.05-or-half-unit bar on the pinhole fixture"
    - "calibrated answers miss ground truth by {metric_answer_rel_error} on average (unit rounding only) while the retired x10 path executed on the same scene misses by {retired_x10_rel_error} (~9x)"
    - "{localize_roundtrip_rate:%} of emitted <point> tags, grounded and un-grounded, parse through the localize stage's exact regex"
    - "{perspective_flip_accuracy:%} of egocentric left/right answers flip correctly across the 180-degree-yaw second camera"
    - "{grounded_label_coverage:%} of fixture matches ground to object labels instead of 'feature', with mean surfaced inlier_ratio {inlier_ratio_mean}"
    - "{unusable_extrinsics_accept_rate} of unusable extrinsics (None, <2 frames, NaN, rotation-only) are wrongly accepted by parse_extrinsics"
    - "real clip (unstubbed VGGT+DepthPro): landmark registration spread ratio {real_registration_spread_ratio}, objects masked in >=2 frames merged into one cloud {real_object_merge_rate:%}, per-object depth_conf non-constant {real_depth_conf_variation_rate:%}, scale factor within one decade {real_scale_sanity_rate:%}, calibrated distances within 50% of the DepthPro anchor {real_distance_plausibility_rate:%}"
    - "S=1 parity: {s1_message_count} messages, fingerprint {s1_message_hash_crc32} - the legacy branch and the fused run forced to S=1 print the shared integer when identical, -1 on divergence"
    - "{fused_new_family_message_count} perspective + grounded-correspondence messages emitted by the fused run on the real frames"
  establishes:
    - "distance/size QA is metric-calibrated on known geometry: {metric_answer_within_bar_rate:%} of answers within the bar while the retired x10 path fails it by {retired_x10_rel_error} on the same scene"
    - "the single-image S=1 message path is preserved byte-for-byte ({s1_message_hash_crc32} shared fingerprint over {s1_message_count} messages)"
    - "S>=2 fusion on a real clip is registered tightly enough for metric QA ({real_registration_spread_ratio}) with merged objects ({real_object_merge_rate:%}), retained varying depth_conf ({real_depth_conf_variation_rate:%}), sane scale ({real_scale_sanity_rate:%}) and DepthPro-plausible distances ({real_distance_plausibility_rate:%})"
    - "perspective QA flips with viewpoint ({perspective_flip_accuracy:%}) and correspondences are grounded to objects ({grounded_label_coverage:%}) without breaking the localize round-trip ({localize_roundtrip_rate:%})"
  does_not_establish:
    - "any downstream VLM training gain from the emitted data (no model is trained here)"
    - "calibration or fusion quality beyond the synthetic fixture and the single real clip measured"
    - "throughput, cost or memory profile of the fused pipeline"
  not_measured:
    - "dataset-scale acceptance rate and coverage (one real clip, first N sorted frames)"
    - "wall-clock throughput of the VGGT+DepthPro stack"
    - "human plausibility of the new QA families"
  caveat: "The PR's fused real-VGGT path shipped no number (CPU tests verified coerce/back-compat only), so its bars here are derived thresholds on one clip, not published rows."
  next: "Run the fused pipeline over a multi-clip corpus and fine-tune a VLM on the emitted data to tie calibration to spatial-benchmark gains."
provenance:
  baseline: "user_guidance"
  suite: "repo_file:eval/eval_vqasynth_spatial_quality.ipynb"
  scorer: "user_guidance"
  metric_answer_within_bar_rate: "maintainer_comment:@smellslikeml"
  localize_roundtrip_rate: "repo_file:tests/test_correspondence.py"
  perspective_flip_accuracy: "repo_file:tests/test_prompts.py"
  metric_answer_rel_error: "repo_file:tests/test_metric_calibration.py"
  retired_x10_rel_error: "repo_file:tests/test_metric_calibration.py"
  grounded_label_coverage: "repo_file:tests/test_correspondence_grounding.py"
  inlier_ratio_mean: "repo_file:tests/test_correspondence_grounding.py"
  unusable_extrinsics_accept_rate: "repo_file:tests/test_prompts.py"
  real_registration_spread_ratio: "user_guidance"
  real_object_merge_rate: "user_guidance"
  real_depth_conf_variation_rate: "user_guidance"
  real_scale_sanity_rate: "user_guidance"
  real_distance_plausibility_rate: "user_guidance"
  s1_message_hash_crc32: "user_guidance"
  s1_message_count: "user_guidance"
  fused_new_family_message_count: "user_guidance"
  held_constant: "user_guidance"
  compute: "user_guidance"
loop:
  max_iterations: 8
  fix_code: true
```